# 10. Machine Learning: Zero-Leakage Performance Regressor
## Gradient Boosting vs Naive Baseline
- Uses strictly t0 pre-publish features (title linguistic properties, duration, cyclical upload hour, baseline subscriber power).
- Target: 7-day log1p(views).
- Evaluates MAE reduction over Naive Median Baseline.


In [ ]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
conn = sqlite3.connect(os.path.join(PROJECT_ROOT, "youtube_growth.db"))

df_features = pd.read_sql_query("SELECT * FROM feature_store", conn)
print("Feature Store Records:", len(df_features))
df_features.head()


In [ ]:
feat_cols = [c for c in df_features.columns if c not in ['feature_id', 'video_id', 'feature_set_version', 'is_target_available', 'extracted_at', 'raw_views']]
X = df_features[feat_cols].values
y = df_features['raw_views'].values

y_log = np.log1p(y)

# Baseline
median_pred = np.full_like(y, np.median(y))
baseline_mae = mean_absolute_error(y, median_pred)

# Model
model = GradientBoostingRegressor(n_estimators=50, max_depth=3, learning_rate=0.08, random_state=42)
model.fit(X, y_log)

preds = np.expm1(model.predict(X))
model_mae = mean_absolute_error(y, preds)
model_r2 = r2_score(y, preds)

print(f"Baseline Naive MAE: {baseline_mae:.2f} views")
print(f"Gradient Boosting MAE: {model_mae:.2f} views")
print(f"MAE Error Reduction: {((baseline_mae - model_mae) / baseline_mae) * 100:.2f}%")
print(f"Model R^2: {model_r2:.4f}")
